# Asset Management: TEMPLATE FOR STRATEGY DEVELOPMENT


### (Not submission, just for standarization and make it easier to combine)







.

## Strategy: FILL YOUR STRATEGY
## Student:  STUDENT NUMBER







.

## 1. Data preparation

In [1]:
import pandas as pd

base = "C:/Users/stenk/OneDrive/Desktop/Asset management/Data/"

annual_data  = pd.read_csv(base + "DE_AT_CH_data_annual.csv")
monthly_data = pd.read_csv(base + "DE_AT_CH_data_monthly.csv")
firms        = pd.read_csv(base + "DE_AT_CH_firms.csv")
factors      = pd.read_csv(base + "Europe_FF_Factors.csv")

print(annual_data.head())
print(monthly_data.head())
print(factors.head())

print(monthly_data.info())  
print(firms.info())

           ISIN  fyear    BEME      OP     INV
0  AT000000STR1   2007  0.5038  0.2386  0.3947
1  AT000000STR1   2008  1.5017  0.2867  0.2589
2  AT000000STR1   2009  1.2163  0.3091 -0.0153
3  AT000000STR1   2010  1.2520  0.2903  0.0726
4  AT000000STR1   2011  1.1162  0.3335  0.0044
           ISIN   mdate     RET   RET11        ME       b       h       s  \
0  DE000A11QW68  200606 -0.0264  0.3490  100.6032  1.7711  1.9829  2.2792   
1  DE000A11QW68  200607 -0.1719 -0.0728   83.3189  1.9704  1.3950  2.1592   
2  DE000A11QW68  200608 -0.0005 -0.2091   83.2671  2.0917  0.5516  2.4282   
3  DE000A11QW68  200609 -0.1441 -0.1636   71.2674  2.1569  1.4690  2.0897   
4  DE000A11QW68  200610 -0.4920 -0.5568   36.1973  1.7767  2.1637  2.2076   

     ivol  
0  0.0243  
1  0.0147  
2  0.0298  
3  0.0166  
4  0.0674  
    mdate   MktRF     SMB     HML      RF     WML
0  199106 -0.0741  0.0061 -0.0069  0.0042  0.0054
1  199107  0.0530 -0.0323 -0.0035  0.0049  0.0506
2  199108  0.0142 -0.0093 -0.0008

## Import and loading  (Remember to copy as path from your location file)

In [2]:
import pandas as pd
import numpy as np

# Set display options (nice for debugging)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Base path
base = r"C:/Users/stenk/OneDrive/Desktop/Asset management/Data/"

# Load data
annual_data  = pd.read_csv(base + "DE_AT_CH_data_annual.csv")
monthly_data = pd.read_csv(base + "DE_AT_CH_data_monthly.csv")
firms        = pd.read_csv(base + "DE_AT_CH_firms.csv")
factors      = pd.read_csv(base + "Europe_FF_Factors.csv")

# Quick inspection
print("ANNUAL")
print(annual_data.head(), "\n")

print("MONTHLY")
print(monthly_data.head(), "\n")

print("FACTORS")
print(factors.head(), "\n")

print("MONTHLY INFO")
print(monthly_data.info())

print("FIRMS INFO")
print(firms.info())

ANNUAL
           ISIN  fyear    BEME      OP     INV
0  AT000000STR1   2007  0.5038  0.2386  0.3947
1  AT000000STR1   2008  1.5017  0.2867  0.2589
2  AT000000STR1   2009  1.2163  0.3091 -0.0153
3  AT000000STR1   2010  1.2520  0.2903  0.0726
4  AT000000STR1   2011  1.1162  0.3335  0.0044 

MONTHLY
           ISIN   mdate     RET   RET11        ME       b       h       s    ivol
0  DE000A11QW68  200606 -0.0264  0.3490  100.6032  1.7711  1.9829  2.2792  0.0243
1  DE000A11QW68  200607 -0.1719 -0.0728   83.3189  1.9704  1.3950  2.1592  0.0147
2  DE000A11QW68  200608 -0.0005 -0.2091   83.2671  2.0917  0.5516  2.4282  0.0298
3  DE000A11QW68  200609 -0.1441 -0.1636   71.2674  2.1569  1.4690  2.0897  0.0166
4  DE000A11QW68  200610 -0.4920 -0.5568   36.1973  1.7767  2.1637  2.2076  0.0674 

FACTORS
    mdate   MktRF     SMB     HML      RF     WML
0  199106 -0.0741  0.0061 -0.0069  0.0042  0.0054
1  199107  0.0530 -0.0323 -0.0035  0.0049  0.0506
2  199108  0.0142 -0.0093 -0.0008  0.0046  0.0377

## Panel Structure

In [3]:
# Check duplicates in annual data
annual_dupes = annual_data.duplicated(subset=["ISIN", "fyear"]).sum()
print("Annual duplicates:", annual_dupes)

# Check duplicates in monthly data
monthly_dupes = monthly_data.duplicated(subset=["ISIN", "mdate"]).sum()
print("Monthly duplicates:", monthly_dupes)

# Number of firms
print("Unique firms (annual):", annual_data["ISIN"].nunique())
print("Unique firms (monthly):", monthly_data["ISIN"].nunique())

# Time coverage
print("Monthly date range:", monthly_data["mdate"].min(), "to", monthly_data["mdate"].max())
print("Annual year range:", annual_data["fyear"].min(), "to", annual_data["fyear"].max())


Annual duplicates: 0
Monthly duplicates: 0
Unique firms (annual): 1546
Unique firms (monthly): 1546
Monthly date range: 199106 to 202006
Annual year range: 1990 to 2019


## Fix dates (YYYYMM → month-end datetime) &  Notebook Block 4 — Merge factors into the monthly panel

In [4]:
import pandas as pd

# Convert YYYYMM integers like 200606 -> Timestamp('2006-06-30')
monthly_data["date"] = pd.to_datetime(monthly_data["mdate"].astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
factors["date"]      = pd.to_datetime(factors["mdate"].astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)

# Sort for clean panel operations later
monthly_data = monthly_data.sort_values(["ISIN", "date"]).reset_index(drop=True)
factors      = factors.sort_values(["date"]).reset_index(drop=True)

# Quick sanity check
print(monthly_data[["ISIN", "mdate", "date"]].head())
print(factors[["mdate", "date"]].head())
print("Monthly date range:", monthly_data["date"].min(), "to", monthly_data["date"].max())
print("Factors date range:", factors["date"].min(), "to", factors["date"].max())

monthly = monthly_data.merge(
    factors[["date", "MktRF", "SMB", "HML", "RF", "WML"]],
    on="date",
    how="left",
    validate="m:1"   # many firms per month -> one factor row per month
)

# Merge diagnostics
missing = monthly[["MktRF","SMB","HML","RF","WML"]].isna().any(axis=1).mean()
print(f"Share of firm-month rows missing any factor: {missing:.4%}")

# Show a few rows
print(monthly.head())
print(monthly.info())

bad_months = monthly.loc[monthly["RF"].isna(), "date"].drop_duplicates().sort_values()
print("Unmatched months (sample):")
print(bad_months.head(20))
print("Count unmatched months:", bad_months.shape[0])


           ISIN   mdate       date
0  AT000000STR1  200808 2008-08-31
1  AT000000STR1  200809 2008-09-30
2  AT000000STR1  200810 2008-10-31
3  AT000000STR1  200811 2008-11-30
4  AT000000STR1  200812 2008-12-31
    mdate       date
0  199106 1991-06-30
1  199107 1991-07-31
2  199108 1991-08-31
3  199109 1991-09-30
4  199110 1991-10-31
Monthly date range: 1991-06-30 00:00:00 to 2020-06-30 00:00:00
Factors date range: 1991-06-30 00:00:00 to 2020-06-30 00:00:00
Share of firm-month rows missing any factor: 0.0000%
           ISIN   mdate     RET   RET11         ME       b       h       s    ivol       date   MktRF     SMB     HML  \
0  AT000000STR1  200808 -0.0787 -0.1853  7079.2492  1.3861  0.2508  0.6964  0.0208 2008-08-31 -0.0457  0.0011 -0.0009   
1  AT000000STR1  200809 -0.2988 -0.4384  4963.8915  1.5177  0.1330  0.7892  0.0283 2008-09-30 -0.1507 -0.0310  0.0091   
2  AT000000STR1  200810 -0.5557 -0.7252  2205.9514  1.5870  0.0412  1.5201  0.0519 2008-10-31 -0.2202 -0.0491 -0.0353   
3

# 2. Strategy Definition

## Parameters: 
(personalize as needed, reference to the variable names as before)

In [5]:
# =========================
# Strategy specification
# =========================

# Universe
UNIVERSE_DESC = "German/Austrian/Swiss equities (ISINs in DE/AT/CH), monthly data"
START_YYYYMM  = 199106
END_YYYYMM    = 202006

# Rebalance
REBALANCE_FREQ = "M"   # monthly or quarterly or yearly if you want

# Signal
SIGNAL_NAME = 
SIGNAL_DESC = 
)

# Construction
EQUAL_WEIGHT_BY_GEO = # True or False
MIN_WEIGHT_PER_STOCK = 
MAX_WEIGHT_PER_STOCK = 

# Constraints

# Add constraints like value or other factors

# Timing / lags
SIGNAL_LAG_MONTHS =    # signal at month t -> positions/returns realized in month t+1

# Risk management inputs

# Depends on strategy




SyntaxError: unmatched ')' (493464128.py, line 16)

## Signals

# 3. Bactkest and Plotting

# 4. Performance: 